# Dataset Exploration

In [ ]:
from pathlib import Path
import sys

import polars as pl
import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

TRAIN_PATH = ROOT / "src" / "data" / "train.parquet"
TEST_PATH = ROOT / "src" / "data" / "test.parquet"
OUTPUT_DIR = ROOT / "outputs" / "notebook_exploration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"root={ROOT}")
print(f"train={TRAIN_PATH.exists()} -> {TRAIN_PATH}")
print(f"test={TEST_PATH.exists()} -> {TEST_PATH}")


## Parquet Metadata

This checks row counts, column counts, and file size without reading the full dataset into memory.


In [ ]:
for path in [TRAIN_PATH, TEST_PATH]:
    parquet_file = pq.ParquetFile(path)
    size_mb = path.stat().st_size / 1024**2
    print(f"{path.name}: {parquet_file.metadata.num_rows:,} rows, {parquet_file.metadata.num_columns:,} columns, {size_mb:,.1f} MB")


## Schema Validation

Use the repository validation function to confirm that the train and test files match the expected competition structure.


In [ ]:
from src.data.splits import load_data, validate_schema

train_df, test_df = load_data(TRAIN_PATH, TEST_PATH)
validate_schema(train_df, test_df)
print("schema ok")
print(f"train shape: {train_df.shape}")
print(f"test shape:  {test_df.shape}")


## Column Overview


In [ ]:
feature_cols = [c for c in train_df.columns if c.startswith("feature_")]
id_cols = ["id", "code", "sub_code", "sub_category", "horizon", "ts_index"]

print(f"feature columns: {len(feature_cols)}")
print("first features:", feature_cols[:10])
print("train-only columns:", sorted(set(train_df.columns) - set(test_df.columns)))
print("test-only columns:", sorted(set(test_df.columns) - set(train_df.columns)))


## Horizon And Time Coverage


In [ ]:
horizon_summary = (
    train_df
    .group_by("horizon")
    .agg(
        pl.len().alias("rows"),
        pl.col("ts_index").min().alias("min_ts"),
        pl.col("ts_index").max().alias("max_ts"),
        pl.col("y_target").mean().alias("target_mean"),
        pl.col("y_target").std().alias("target_std"),
        pl.col("weight").mean().alias("weight_mean"),
    )
    .sort("horizon")
)
horizon_summary


In [ ]:
test_horizon_summary = (
    test_df
    .group_by("horizon")
    .agg(
        pl.len().alias("rows"),
        pl.col("ts_index").min().alias("min_ts"),
        pl.col("ts_index").max().alias("max_ts"),
    )
    .sort("horizon")
)
test_horizon_summary


## Missing Values In A Small Sample



In [ ]:
SAMPLE_ROWS = 100_000

sample = train_df.sample(n=min(SAMPLE_ROWS, train_df.height), seed=42)
missing_sample = (
    sample
    .select([pl.col(c).is_null().mean().alias(c) for c in feature_cols])
    .transpose(include_header=True, header_name="feature", column_names=["missing_rate"])
    .sort("missing_rate", descending=True)
)
missing_sample.head(20)


## Target Distribution By Horizon

The metric is sensitive to heavy-tail rows. These quantiles give a quick view of target scale by horizon.


In [ ]:
target_quantiles = (
    train_df
    .group_by("horizon")
    .agg(
        pl.col("y_target").quantile(0.01).alias("q01"),
        pl.col("y_target").quantile(0.10).alias("q10"),
        pl.col("y_target").median().alias("q50"),
        pl.col("y_target").quantile(0.90).alias("q90"),
        pl.col("y_target").quantile(0.99).alias("q99"),
    )
    .sort("horizon")
)
target_quantiles


## Simple Temporal Split Check


In [ ]:
from src.data.splits import build_time_split

train_mask, val_mask, val_start_ts = build_time_split(train_df, val_ratio=0.15)
print(f"validation starts at ts_index={val_start_ts}")
print(f"fit rows={int(train_mask.sum()):,}")
print(f"validation rows={int(val_mask.sum()):,}")


## Baseline Command

Run the quick baseline from a terminal, or set `RUN_BASELINE = True` below to launch it from the notebook.


In [ ]:
import subprocess

RUN_BASELINE = False
cmd = [sys.executable, str(ROOT / "scripts" / "run_baselines.py")]
print(" ".join(cmd))

if RUN_BASELINE:
    subprocess.run(cmd, cwd=ROOT, check=True)
